# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

#### K-Means Clustering

**Why K-Means?**
- K-Means is interpretable: cluster centers show the archetype profile
- Performance metric is directly computable: silhouette score + visual inspection of cluster-score separation

**Why not alternatives?**
- DBSCAN (density-based): unclear density thresholds for content performance, risk of noise cluster
- Hierarchical: slower on 182K items, hard to explain dendrograms to stakeholders
- Gaussian Mixture: more parameters to tune with no semantic gain

**Features for clustering** (standardized):
- `ctr` (actual CTR, 0–100 scale after converting from decimal)
- `avg_position` (search rank, 1–100+)
- `engagement_rate` (from starter CSV, proxy for scroll/dwell)
- `scroll_rate` (from starter CSV)
- `impressions_log` = log10(impressions + 1) — compress high-volume tail

All features will be StandardScaler normalized before clustering.

**Expected archetypes** (speculative, to validate against):
- Champion: high CTR, top-5 position, high engagement
- Rising Star: lower position but climbing, moderate-high volume
- Title/CTR Problem: good position, low CTR (matches baseline high-score picks)
- Niche/Low Volume: low impressions, variable engagement
- Dead Weight: very low CTR, low position, low volume

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Why grouped split (by client)?

Content items within a single client are more similar to each other than random items across clients. If we split randomly by rows, K-Means will fit clusters that may just memorize client-specific behavior. A grouped split by client_hash_id tests: "Does the cluster structure from one client's content apply to another client?"

This prevents data leakage in the sense that we're not letting one client's behavior dominate the cluster centroids.

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.pyplot as plt
import os

os.makedirs('work/outputs', exist_ok=True)

df = pd.read_csv('work/outputs/baseline_action_score.csv')

print(f"Loaded {len(df):,} scored items from baseline")
print(f"Columns: {df.columns.tolist()}")
print(f"\nAction label breakdown:")
print(df['action_label'].value_counts())

Loaded 182,135 scored items from baseline
Columns: ['content_hash_id', 'client_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'days_active', 'first_date', 'last_date', 'position_bucket', 'expected_ctr', 'ctr_gap', 'score', 'reason_code', 'action_label']

Action label breakdown:
action_label
NO_ACTION_CTR_ON_TRACK          160982
FIX_TITLE_META_MONITOR           15882
FIX_TITLE_META_HIGH_PRIORITY      5271
Name: count, dtype: int64


In [18]:
# Define clustering features
features_for_clustering = ['ctr', 'avg_position']

# Add log-impressions to compress high-volume tail
df['impressions_log'] = np.log10(df['impressions'] + 1)
features_for_clustering.append('impressions_log')

print(f"\nClustering features available: {features_for_clustering}")
print(f"(Note: engagement_rate and scroll_rate not in warehouse baseline output)")

print(f"\nFeature summary (before standardization):")
print(df[features_for_clustering].describe())

# Check for missing values
print(f"\nMissing values per feature:")
for col in features_for_clustering:
    pct_missing = 100 * df[col].isna().sum() / len(df)
    print(f"  {col}: {pct_missing:.2f}%")

# Fill any missing values
for col in features_for_clustering:
    df[col] = df[col].fillna(df[col].median())


Clustering features available: ['ctr', 'avg_position', 'impressions_log']
(Note: engagement_rate and scroll_rate not in warehouse baseline output)

Feature summary (before standardization):
                 ctr   avg_position  impressions_log
count  182135.000000  182135.000000    182135.000000
mean        0.003172      18.002448         3.253352
std         0.005406      14.707742         0.782179
min         0.000000       0.000000         2.004321
25%         0.000000       7.600625         2.608526
50%         0.001716      12.772799         3.180126
75%         0.004090      23.440158         3.819215
max         0.536745     173.269278         6.462790

Missing values per feature:
  ctr: 0.00%
  avg_position: 0.00%
  impressions_log: 0.00%


In [19]:
# Grouped train/test split

# Get unique clients
unique_clients = df['client_hash_id'].unique()
print(f"Total unique clients: {len(unique_clients)}")

# 80/20 split by client (grouped split)
np.random.seed(42)
np.random.shuffle(unique_clients)

split_idx = int(0.8 * len(unique_clients))
train_clients = set(unique_clients[:split_idx])
test_clients = set(unique_clients[split_idx:])

# Create train/test masks
train_mask = df['client_hash_id'].isin(train_clients)
test_mask = df['client_hash_id'].isin(test_clients)

print(f"\nTrain/test split (by client):")
print(f"  Train: {len(train_clients)} clients, {train_mask.sum():,} items")
print(f"  Test:  {len(test_clients)} clients, {test_mask.sum():,} items")

# Extract feature matrices
X_full = df[features_for_clustering].values
X_train = X_full[train_mask]
X_test = X_full[test_mask]

print(f"\nFeature matrix shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")

# Standardize BOTH on train mean/std (prevent test leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_full_scaled = scaler.transform(X_full)

print(f"\n✓ Features standardized (mean=0, std=1)")
print(f"  Train scaled mean: {X_train_scaled.mean(axis=0)}")
print(f"  Train scaled std: {X_train_scaled.std(axis=0)}")

Total unique clients: 59

Train/test split (by client):
  Train: 47 clients, 161,153 items
  Test:  12 clients, 20,982 items

Feature matrix shapes:
  X_train: (161153, 3)
  X_test: (20982, 3)

✓ Features standardized (mean=0, std=1)
  Train scaled mean: [ 1.08497831e-13 -1.81987057e-14 -5.03078031e-13]
  Train scaled std: [1. 1. 1.]


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.